# Model Insights
This notebook houses all the work needed to generate the model insights displayed with the prediction tool itself. This includes: PDP's, SHAP values, Residual analysis.

In [1]:
!pip install shap

In [2]:
import pandas as pd
import numpy as np
import pickle
from sklearn.inspection import partial_dependence
import os
import shap

In [3]:
# Loading the winning model pipeline and training/testing data
with open('../deployment/model_artifacts_002.pkl', 'rb') as f:
    artifacts = pickle.load(f)

best_model = artifacts['model']
train_df = pd.read_csv('../data/train_data.csv')
test_df = pd.read_csv('../data/test_data.csv')
X_train = train_df.drop(columns='Sold_Price')
X_test = test_df.drop(columns='Sold_Price')

## Partial dependency plots

In [4]:
# Selecting the continuous variables to visualize in PDP's
# Categorical vars/Target Encoded features are less interpretable on a PDP line chart
features_to_plot = [
    'Mileage', 'car_age', 'Engine_Displacement_L', 'mileage_per_year', "Gears"
]
pdp_results = []

# Fill NaNs temporarily so scikit-learn can calculate the percentiles
X_train_pdp = X_train.copy()
X_train_pdp['mileage_per_year'] = X_train_pdp['mileage_per_year'].fillna(X_train_pdp['mileage_per_year'].median())

print("Calculating Partial Dependency Plots...")
for feature in features_to_plot:
    pdp = partial_dependence(best_model, X_train_pdp, features=[feature], grid_resolution=50)
    
    # Accommodate scikit-learn version differences for the grid key
    grid_key = 'values' if 'values' in pdp else 'grid_values'
    grid_values = pdp[grid_key][0]
    
    # The model targets log-space, inverse transform to real dollars
    avg_predictions_dollars = np.expm1(pdp['average'][0])
    
    for val, pred in zip(grid_values, avg_predictions_dollars):
        pdp_results.append({
            'Feature': feature,
            'Feature_Value': val,
            'Predicted_Price': pred
        })

Calculating Partial Dependency Plots...


In [5]:
# Exporting PDP data to the frontend data directory
pdp_df = pd.DataFrame(pdp_results)
pdp_df.to_csv('../data/frontend_data/pdp_data.csv', index=False)
print("Saved PDP data to ../data/frontend_data/pdp_data.csv")

Saved PDP data to ../data/frontend_data/pdp_data.csv


## Residual Analysis

In [6]:
# Moving to the residual data
df_residuals = X_test.copy()

# Original target in dollars
y_test_dollars = test_df['Sold_Price']

# Predictions in log space
y_pred_log = best_model.predict(X_test)

# Predictions in dollars 
y_pred_dollars = np.expm1(y_pred_log)

# Creating df with target and prediction
df_residuals['Sold_Price'] = y_test_dollars.values
df_residuals['Predicted_Price'] = y_pred_dollars

df_residuals.head()

,Make,Model,Mileage,Exterior Color,Interior Color,Gears,Engine_Displacement_L,flaw_count,2_keys_ind,owners_manual_ind,...,auction_month_10,auction_month_11,auction_month_12,trim_tier_base,trim_tier_economy,trim_tier_high_performance,trim_tier_sport_premium,trim_tier_ultra_premium,Sold_Price,Predicted_Price
0,BMW,3 Series,10.797954,5,3,6.0,3.0,3,1,1,...,False,False,False,True,False,False,False,False,8250.0,10041.146484
1,BMW,X7,8.612685,13,1,8.0,3.0,2,1,1,...,False,False,False,True,False,False,False,False,65500.0,55442.910156
2,Bentley,Continental,10.385945,13,12,8.0,6.0,1,0,0,...,False,False,False,True,False,False,False,False,58111.0,41392.867188
3,Mazda,ND Miata,8.434029,5,1,6.0,2.0,1,1,1,...,False,False,False,False,True,False,False,False,25250.0,29483.451172
4,Jaguar,XJR,10.491302,1,12,5.0,4.0,3,0,1,...,False,False,False,True,False,False,False,False,15000.0,9816.307617


In [7]:
# Back-calculating the car's Year
if 'Year' not in df_residuals.columns and 'auction_year' in df_residuals.columns and 'car_age' in df_residuals.columns:
    df_residuals['Year'] = df_residuals['auction_year'] - df_residuals['car_age']

# Keeping only the columns needed for the frontend visualization to reduce file size
cols_to_keep = ['Make', 'Model', 'Year', 'Sold_Price', 'Predicted_Price']
available_cols = [c for c in cols_to_keep if c in df_residuals.columns]

os.makedirs('../data/frontend_data', exist_ok=True)
df_residuals[available_cols].to_csv('../data/frontend_data/residual_data.csv', index=False)

## Global SHAP values

In [8]:
# Compute global SHAP feature importance on the test set
preprocessor = best_model.named_steps['preprocessor']
xgb_model = best_model.named_steps['xgb']

X_test_transformed = preprocessor.transform(X_test)
feature_names = [n.split('__')[-1] for n in preprocessor.get_feature_names_out()]

explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test_transformed)

# Mean absolute SHAP value per feature = global importance
shap_importance = pd.DataFrame({
    'feature': feature_names,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False).head(20)

shap_importance.to_csv('../data/frontend_data/shap_importance.csv', index=False)
print("Saved SHAP importance to ../data/frontend_data/shap_importance.csv")
print(shap_importance)

Saved SHAP importance to ../data/frontend_data/shap_importance.csv
                        feature  mean_abs_shap
4     make_model_mileage_bucket       0.229275
3               make_model_year       0.096553
5               make_model_trim       0.084986
31            car_age_x_mileage       0.078637
11        Engine_Displacement_L       0.061504
1                         Model       0.056027
47            text_component_11       0.052861
7                       Mileage       0.047721
12                   flaw_count       0.036255
68  Transmission_Type_Automatic       0.032909
53            text_component_17       0.031394
27                      car_age       0.027057
26                   model_year       0.026071
30                  SP500_Close       0.021876
38             text_component_2       0.021437
37             text_component_1       0.020897
10                        Gears       0.019830
33         sp500_x_auction_year       0.018808
39             text_component_3       0.